In [5]:
# You only need to run this block once per session to install Gurobi (and other libraries)

%pip install pyomo gurobipy pandas

In [ ]:
# v0.2.1 - 9/10/26
# Implement metric params, guessing we want max weekends worked as metric

import gurobipy as gp
from gurobipy import GRB, Model

import pandas as pd
from typing import Any
import json

from datetime import datetime, date as Date
import calendar
from IPython.display import HTML

# =================================
# PARAMETERS
# =================================
YEAR = 2026
MONTH = 1

residents = [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
    "G",
]

vacation_requests = {
    "A": ["2026-01-03", "2026-01-14", "2026-01-24"],
    "B": ["2026-01-03", "2026-01-15", "2026-01-25"],
    "C": ["2026-01-03", "2026-01-16", "2026-01-26"],
    "D": ["2026-01-03", "2026-01-17", "2026-01-27"],
    "E": ["2026-01-03", "2026-01-18", "2026-01-28"],
    "F": ["2026-01-03", "2026-01-19", "2026-01-29"],
    "G": ["2026-01-03", "2026-01-20", "2026-01-30"],
}

metrics = {
    "vacations_denied": {"lb": 0, "ub": 99, "weight": 1},
    "test": {"lb": 0, "ub": 99, "weight": 0},
}


# =================================
# UTILITIES
# =================================
def is_weekend(date_str: str) -> bool:
    return datetime.strptime(date_str, "%Y-%m-%d").weekday() >= 5  # Saturday (5) or Sunday (6)


# =================================
# GENERATE
# =================================
first_date = Date(YEAR, MONTH, 1)
last_day = calendar.monthrange(YEAR, MONTH)[1]
last_date = Date(YEAR, MONTH, last_day)

# Create list of dates, starting from the first date to the last date, in ISO8601 string format
dates : list[Date] = (
    pd.date_range(start=first_date, end=last_date)
    .strftime("%Y-%m-%d")
    .tolist()
)

# =================================
# MODEL PARAMETERS
# =================================
m: Model = Model()
# m.setParam('LogToConsole', 0)


# =================================
# BUILD 
# =================================

# --------- Decision variables ----------
x_rd = m.addVars(len(residents), len(dates), 
                    lb=0, ub=1, 
                    vtype=GRB.BINARY)

# Set variable names
for (r, resident) in enumerate(residents):
    for (d, date) in enumerate(dates):
        x_rd[r, d].VarName = f"{resident}_{date}"


# --------- Metric variables ----------
for metric_name, metric in metrics.items():
    metric["var"] = m.addVar(lb=metric["lb"], ub=metric["ub"], 
                        vtype=GRB.INTEGER)
    metric["var"].VarName = metric_name


# --------- Constraints ----------
# One res for each date
for (d, date) in enumerate(dates):
    m.addConstr(gp.quicksum(x_rd[r, d] for (r, resident) in enumerate(residents)) == 1,
                name=f"single_shift_{date}")

# Resident coverage ub of 7 across the entire month
for (r, resident) in enumerate(residents):
    m.addConstr(gp.quicksum(x_rd[r, d] for (d, date) in enumerate(dates)) <= 7,
                name=f"coverage_ub_{resident}")


# --------- Metric Constraints ----------
m.addConstr(
    metrics["vacations_denied"]["var"] >= gp.quicksum(
        x_rd[r, d]
        for r, resident in enumerate(residents)
        for d, shift_date in enumerate(dates)
        if shift_date in vacation_requests[resident]
    ),
    name="metric_vacations_denied"
)

for (r, resident) in enumerate(residents):
    m.addConstr(
        metrics["vacations_denied"]["var"] >= gp.quicksum(
            x_rd[r, d]
            for d, shift_date in enumerate(dates)
            if is_weekend(shift_date)
        ),
        name=f"metric_weekends_worked_r_{resident}"
    )




# --------- Objective function ----------
# Obj fxn: minimize number of assignments to resident 0 (i.e., the first resident, "A")
# m.setObjective(
#     gp.quicksum(x_rd[r, d] for r, d in x_rd if r == 0),
#     GRB.MINIMIZE
# )

m.setObjective(
    gp.quicksum(metric["weight"] * metric["var"] for metric in metrics.values()),
    GRB.MINIMIZE
)


# =================================
# SOLVE
# =================================
m.optimize()
tolerance = 0.5

# Extract solution variable as boolean
x_rd_sol = [[x_rd[i, j].X > tolerance for j in range(len(dates))]
          for i in range(len(residents))]

# =================================
# REPORT
# =================================

# ---------- Deubgging ----------
# print vars
# for v in m.getVars():
#     print(f"{v.VarName} = {v.X}")

# infeasible: bool = m.Status == GRB.INFEASIBLE
# print(m.Status)
current = datetime.now().strftime("%m-%d-%Y-%H-%M-%S")
m.write(f"output/{current}-model.lp")


# ---------- Table ----------
table_rows = []

for r, resident in enumerate(residents):
    total_shifts = sum(
        x_rd_sol[r][d]
        for d in range(len(dates))
    )

    weekend_shifts = sum(
        x_rd_sol[r][d]
        for d, shift_date in enumerate(dates)
        if is_weekend(shift_date)
    )

    denied_requests = sum(
        x_rd_sol[r][d]
        for d, shift_date in enumerate(dates)
        if shift_date in vacation_requests[resident]
    )

    requested_vacations = len(vacation_requests[resident])

    table_rows.append({
        "Resident": resident,
        "Total Shifts": total_shifts,
        "Weekend Shifts": weekend_shifts,
        "Vacation Requests": requested_vacations,
        "Requests Denied": denied_requests,
    })

table_rows_df = pd.DataFrame(table_rows)
display(table_rows_df)

# ---------- Calendar ----------
class Schedule(calendar.HTMLCalendar):
    def __init__(self, event_names, dates, occurs):
        # initialize calendar.HTMLCalendar to start weeks on Sunday
        super().__init__(calendar.SUNDAY)

        self.assignments = {}
        for r in range(len(residents)):
            for d in range(len(dates)):
                if (x_rd_sol[r][d]):
                    self.assignments[dates[d]] = residents[r]
        

    def formatday(self, day, weekday):
        if day == 0:
            # empty table cell
            return '<td></td>'

        # render cell with day and assignment
        event_date = Date(
            self.current_year,
            self.current_month,
            day
        ).strftime("%Y-%m-%d")
        event = self.assignments.get(event_date, "")

        event_html = (f'<div>{event}</div>') if event else ""
        return (f'<td style="border:1px solid #ffffff; width:80px; height:60px; vertical-align:top">'
                f'<strong>{day}</strong>'
                f'{event_html}'
                f'</td>')

    def formatmonth(self, theyear, themonth, withyear=True):
        self.current_year = theyear
        self.current_month = themonth
        return super().formatmonth(theyear, themonth, withyear)

schedule = Schedule(residents, dates, x_rd_sol)
HTML(schedule.formatmonth(YEAR, MONTH))



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 7 265U, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 14 logical processors, using up to 14 threads

Optimize a model with 46 rows, 219 columns and 526 nonzeros (Min)
Model fingerprint: 0x414c850e
Model has 1 linear objective coefficients
Variable types: 0 continuous, 219 integer (217 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+02]
  RHS range        [1e+00, 7e+00]

Found heuristic solution: objective 4.0000000
Presolve removed 0 rows and 1 columns
Presolve time: 0.00s
Presolved: 46 rows, 218 columns, 519 nonzeros
Variable types: 0 continuous, 218 integer (217 binary)

Root relaxation: objective 1.285714e+00, 54 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Inc

,Resident,Total Shifts,Weekend Shifts,Vacation Requests,Requests Denied
0,A,7,1,3,0
1,B,2,1,3,0
2,C,3,1,3,0
3,D,4,1,3,0
4,E,6,2,3,0
5,F,6,2,3,0
6,G,3,1,3,1
